In [2]:
import requests
import pandas as pd 
from bs4 import BeautifulSoup as BS

In [3]:
url = "https://books.toscrape.com/"
headers = {"User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)" "AppleWebKit/537.36 (KHTML, like Gecko)" "Chrome/127.0.0.0 Safari/537.36")}

In [4]:
books_data = []
books = []

In [5]:
for page_no in range (1, 6):
    if page_no == 1: url_00 = url
    else: url_00 = f"{url}catalogue/page-{page_no}.html"
    print("Scrapping :", url_00)
    reponse = requests.get(url_00, headers=headers)
    print("Status    :",reponse.status_code)

    if (reponse.status_code != 200): 
        print("book not found")
        continue

    soup = BS(reponse.content, "html.parser")
    books.extend(soup.find_all("article", class_="product_pod"))
    print("Book found:", len(books))

Scrapping : https://books.toscrape.com/
Status    : 200
Book found: 20
Scrapping : https://books.toscrape.com/catalogue/page-2.html
Status    : 200
Book found: 40
Scrapping : https://books.toscrape.com/catalogue/page-3.html
Status    : 200
Book found: 60
Scrapping : https://books.toscrape.com/catalogue/page-4.html
Status    : 200
Book found: 80
Scrapping : https://books.toscrape.com/catalogue/page-5.html
Status    : 200
Book found: 100


In [6]:
for book in books:
    title = book.h3.a["title"]
    price = book.find("p",class_="price_color").text.strip()
    rating = book.find("p",class_="star-rating")["class"][1]
    availability = book.find("p",class_="instock availability").text.strip()
    relative_link = book.h3.a["href"]
    full_link = url + "catalogue/" + relative_link.replace("../../../","")
    books_data.append({"title": title,"price": price,"rating":
    rating,"availability": availability,"link": full_link})

In [7]:
df = pd.DataFrame(books_data)
df.head()


,title,price,rating,availability,link
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/catalogue/catalogue...
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/catalogue/catalogue...
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/catalogue/catalogue...
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/catalogue/catalogue...
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/catalogue/catalogue...


In [8]:
print("\nShape")
print(df.shape)
print("\nColumns")
print(df.columns)
print("\nData types")
print(df.dtypes)


Shape
(100, 5)

Columns
Index(['title', 'price', 'rating', 'availability', 'link'], dtype='object')

Data types
title           object
price           object
rating          object
availability    object
link            object
dtype: object


In [9]:
df["price"] = (df["price"].str.replace("£", "", regex=False).astype(float))
rating_map = {"One": 1,"Two": 2,"Three": 3,"Four": 4,"Five": 5}
df["rating"] = df["rating"].map(rating_map)

In [10]:
df.head()

,title,price,rating,availability,link
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/catalogue...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/catalogue...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/catalogue...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/catalogue...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/catalogue...


In [11]:
df.dtypes

title            object
price           float64
rating            int64
availability     object
link             object
dtype: object

In [12]:
df.to_csv("books_dataset.csv",index=False,encoding="utf-8-sig")

In [13]:
print("\nAverage price:")
print(df["price"].mean())
print("\nAverage rating:")
print(df["rating"].mean())
print("\nMost expensive book:")
print(df.loc[df["price"].idxmax(),"title"])
print("\nHighest rated books:")
print(df[df["rating"] == 5][["title", "price", "rating"]].head())


Average price:
34.560700000000004

Average rating:
2.93

Most expensive book:
The Death of Humanity: and the Case for Life

Highest rated books:
                                                title  price  rating
4               Sapiens: A Brief History of Humankind  54.23       5
12                                        Set Me Free  17.46       5
13  Scott Pilgrim's Precious Little Life (Scott Pi...  52.29       5
14                          Rip it Up and Start Again  35.02       5
23                         Chase Me (Paris Nights #2)  25.27       5
